# 04 — Popularity & Sales Factors

Analyse des facteurs influençant la popularité et les ventes de jeux vidéo sur Steam.

**Question BONUS** : What factors affect the popularity or sales of a video game? Take advantage of this opportunity to analyze the video game market globally.

**Objectif métier** : Identifier les leviers de succès pour Ubisoft avant le développement d'un nouveau jeu.

In [0]:
from pyspark.sql import functions as F

df = (spark.table("steam_clean")
    .withColumn("price_tier",
        F.when(F.col("price_usd") == 0, "1_Free")
        .when(F.col("price_usd") <= 5, "2_Budget (1-5$)")
        .when(F.col("price_usd") <= 15, "3_Mid (6-15$)")
        .when(F.col("price_usd") <= 30, "4_Standard (16-30$)")
        .when(F.col("price_usd") <= 60, "5_Premium (31-60$)")
        .otherwise("6_Luxury (60$+)")
    )
    .withColumn("platform_count",
        F.col("platform_windows").cast("int") +
        F.col("platform_mac").cast("int") +
        F.col("platform_linux").cast("int")
    )
    .withColumn("lang_count",
        F.when(F.col("languages").isNotNull(), F.size(F.col("languages"))).otherwise(None)
    )
)
print(f"Total games: {df.count()}")

Total games: 55690


## 1. Distribution overview

Understanding the distribution of our success metrics (owners, reviews, rating) is essential before drilling into individual factors -- extreme skewness affects which summary statistic is meaningful.

**Metrics used throughout:**
- `avg_owners`: popularity proxy (Steam Spy estimate)
- `total_reviews`: engagement proxy
- `rating_ratio`: perceived quality (99.7% coverage)
- `estimated_revenue`: approximate revenue proxy

In [0]:
display(df.select("avg_owners", "total_reviews", "rating_ratio").summary())

summary,avg_owners,total_reviews,rating_ratio
count,55690,55690,55527
mean,133513.55719159634,1712.608044532232,0.7365350137972013
stddev,1849924.5941899035,35687.92295880968,0.2423200060117725
min,10000,0,0.0
25%,10000,7,0.6086956521739131
50%,10000,26,0.7985436893203883
75%,35000,145,0.9249448123620309
max,350000000,6730438,1.0


## 2. Price impact on popularity

Price directly affects the purchase barrier; we check whether cheaper games attract more owners or if premium pricing signals quality.

In [0]:
df_price_impact = (df
    .groupBy("price_tier")
    .agg(
        F.count("*").alias("game_count"),
        F.round(F.avg("avg_owners"), 0).alias("avg_owners"),
        F.round(F.avg("total_reviews"), 0).alias("avg_reviews"),
        F.round(F.avg("rating_ratio"), 3).alias("avg_rating")
    )
    .orderBy("price_tier")
)
display(df_price_impact)

price_tier,game_count,avg_owners,avg_reviews,avg_rating
1_Free,7779,314524.0,3468.0,0.697
2_Budget (1-5$),23498,44369.0,285.0,0.724
3_Mid (6-15$),17750,92806.0,1120.0,0.76
4_Standard (16-30$),5497,296577.0,4814.0,0.765
5_Premium (31-60$),1044,622495.0,14422.0,0.769
6_Luxury (60$+),122,152664.0,2354.0,0.719


Databricks visualization. Run in Databricks to view.

## 3. Genre impact

Genre determines the addressable audience size and competition level -- some genres naturally attract more players.

`genre`: 0.3% excluded (160 NULLs).

In [0]:
df_genres = (df
    .filter(F.col("genre").isNotNull())
    .withColumn("genre_single", F.explode(F.col("genre")))
)

df_genre_success = (df_genres
    .groupBy("genre_single")
    .agg(
        F.count("*").alias("game_count"),
        F.round(F.avg("avg_owners"), 0).alias("avg_owners"),
        F.round(F.avg("total_reviews"), 0).alias("avg_reviews"),
        F.round(F.avg("rating_ratio"), 3).alias("avg_rating"),
        F.round(F.avg("estimated_revenue"), 0).alias("avg_revenue")
    )
    .filter(F.col("game_count") >= 500)
    .orderBy(F.desc("avg_owners"))
)
display(df_genre_success)

genre_single,game_count,avg_owners,avg_reviews,avg_rating,avg_revenue
Massively Multiplayer,1460,661185.0,7478.0,0.629,4061752.0
Free to Play,3393,580029.0,6780.0,0.722,29907.0
Action,23759,201095.0,2717.0,0.73,2473019.0
RPG,9534,183200.0,2381.0,0.731,2850130.0
Strategy,10895,150173.0,1450.0,0.719,1849476.0
Adventure,21431,124114.0,1649.0,0.739,1737937.0
Simulation,10836,123071.0,1659.0,0.694,1732166.0
Sports,2666,120056.0,1378.0,0.698,1181507.0
Utilities,682,99545.0,1148.0,0.699,555433.0
Racing,2155,89234.0,1264.0,0.702,1263796.0


Databricks visualization. Run in Databricks to view.

## 4. Multi-platform effect

Supporting Mac/Linux in addition to Windows may reflect studio maturity and increase audience reach.

In [0]:
df_platform_impact = (df
    .groupBy("platform_count")
    .agg(
        F.count("*").alias("game_count"),
        F.round(F.avg("avg_owners"), 0).alias("avg_owners"),
        F.round(F.avg("total_reviews"), 0).alias("avg_reviews"),
        F.round(F.avg("rating_ratio"), 3).alias("avg_rating")
    )
    .orderBy("platform_count")
)
display(df_platform_impact)

platform_count,game_count,avg_owners,avg_reviews,avg_rating
1,41285,102130.0,1251.0,0.722
2,7599,137186.0,1679.0,0.773
3,6806,319782.0,4547.0,0.783


Databricks visualization. Run in Databricks to view.

## 5. Language support impact

Broader language support expands the addressable market; we test whether more languages correlate with higher ownership.

`languages` / `lang_count`: 10 NULLs excluded (negligible).

In [0]:
df_lang_impact = (df
    .filter(F.col("lang_count").isNotNull())
    .withColumn("lang_tier",
        F.when(F.col("lang_count") == 1, "1_Single")
        .when(F.col("lang_count") <= 5, "2_Few (2-5)")
        .when(F.col("lang_count") <= 10, "3_Good (6-10)")
        .otherwise("4_Extensive (11+)")
    )
    .groupBy("lang_tier")
    .agg(
        F.count("*").alias("game_count"),
        F.round(F.avg("avg_owners"), 0).alias("avg_owners"),
        F.round(F.avg("total_reviews"), 0).alias("avg_reviews"),
        F.round(F.avg("rating_ratio"), 3).alias("avg_rating")
    )
    .orderBy("lang_tier")
)
display(df_lang_impact)

lang_tier,game_count,avg_owners,avg_reviews,avg_rating
1_Single,29655,48332.0,319.0,0.724
2_Few (2-5),15062,79516.0,646.0,0.74
3_Good (6-10),6384,277201.0,3260.0,0.761
4_Extensive (11+),4579,662729.0,12097.0,0.771


Databricks visualization. Run in Databricks to view.

## 6. Temporal trends by release year

Older games have had more time to accumulate owners, so comparing cohorts by release year helps disentangle age effects from true popularity.

In [0]:
# avg_owners is a cumulative snapshot, not an annual flow.
# Dividing by years on market would assume linear acquisition,
# which is false (most sales happen at launch). We compare cohorts
# by release_year directly instead.

df_year_success = (df
    .filter(F.col("release_year").isNotNull())
    .filter(F.col("release_year") >= 2015)
    .groupBy("release_year")
    .agg(
        F.count("*").alias("game_count"),
        F.round(F.avg("avg_owners"), 0).alias("avg_owners"),
        F.round(F.avg("total_reviews"), 0).alias("avg_reviews"),
        F.round(F.avg("rating_ratio"), 3).alias("avg_rating")
    )
    .orderBy("release_year")
)
display(df_year_success)

release_year,game_count,avg_owners,avg_reviews,avg_rating
2015,2575,248384.0,3654.0,0.69
2016,4185,162806.0,2017.0,0.706
2017,6017,123793.0,1698.0,0.7
2018,7678,84989.0,1259.0,0.695
2019,6968,73813.0,997.0,0.728
2020,8305,75845.0,1163.0,0.756
2021,8823,62021.0,689.0,0.766
2022,7455,46911.0,529.0,0.794


Databricks visualization. Run in Databricks to view.

## 7. Publisher effect

Publisher reputation and marketing budget create a visibility advantage that amplifies other success factors.

In [0]:
df_pub_success = (df
    .filter(F.col("publisher").isNotNull())
    .groupBy("publisher")
    .agg(
        F.count("*").alias("game_count"),
        F.round(F.avg("avg_owners"), 0).alias("avg_owners"),
        F.round(F.avg("total_reviews"), 0).alias("avg_reviews"),
        F.round(F.avg("rating_ratio"), 3).alias("avg_rating"),
        F.round(F.sum("estimated_revenue"), 0).alias("total_revenue")
    )
    .filter(F.col("game_count") >= 20)
    .orderBy(F.desc("avg_owners"))
    .limit(15)
)
display(df_pub_success)

publisher,game_count,avg_owners,avg_reviews,avg_rating,total_revenue
Valve,35,2.2939857E7,357381.0,0.891,2.40048E9
"CAPCOM Co., Ltd.",24,2360208.0,39790.0,0.833,1.67480205E9
Bethesda Softworks,55,2042273.0,41789.0,0.793,2.21021685E9
Xbox Game Studios,46,1945978.0,37146.0,0.86,2.25967235E9
Electronic Arts,88,1739602.0,27701.0,0.797,3.188435E9
Warner Bros. Interactive Entertainment,29,1431552.0,16404.0,0.824,6.2244185E8
Ubisoft,127,1358780.0,23613.0,0.776,3.4803004E9
2K,67,1175896.0,12758.0,0.713,1.30179715E9
Bohemia Interactive,28,1113214.0,22137.0,0.681,7.363881E8
Activision,51,985980.0,9433.0,0.782,1.67836075E9


Databricks visualization. Run in Databricks to view.

## 8. Free-to-Play vs Paid

Free-to-Play games attract more players but monetization mechanics may affect perceived quality.

In [0]:
df_model_impact = (df
    .withColumn("business_model",
        F.when(F.col("is_free"), "Free to Play")
        .otherwise("Paid")
    )
    .groupBy("business_model")
    .agg(
        F.count("*").alias("game_count"),
        F.round(F.avg("avg_owners"), 0).alias("avg_owners"),
        F.round(F.avg("total_reviews"), 0).alias("avg_reviews"),
        F.round(F.avg("rating_ratio"), 3).alias("avg_rating"),
        F.round(F.percentile_approx("avg_owners", 0.9), 0).alias("owners_p90")
    )
)
display(df_model_impact)

business_model,game_count,avg_owners,avg_reviews,avg_rating,owners_p90
Paid,47911,104124.0,1428.0,0.743,150000
Free to Play,7779,314524.0,3468.0,0.697,350000


## 9. Combined factors

Individual factors interact -- a premium game with multi-language support on multiple platforms performs differently than each factor alone suggests.

In [0]:
df_success_factors = (df
    .filter(F.col("lang_count").isNotNull())
    .withColumn("has_multi_lang", F.col("lang_count") >= 5)
    .withColumn("has_multi_platform", F.col("platform_count") >= 2)
)

display(
    df_success_factors
    .groupBy("price_tier", "has_multi_lang", "has_multi_platform")
    .agg(
        F.count("*").alias("games"),
        F.round(F.avg("avg_owners"), 0).alias("avg_owners"),
        F.round(F.avg("rating_ratio"), 3).alias("avg_rating")
    )
    .orderBy(F.desc("avg_owners"))
)

price_tier,has_multi_lang,has_multi_platform,games,avg_owners,avg_rating
5_Premium (31-60$),true,true,85,1904176.0,0.79
1_Free,true,true,538,1696171.0,0.685
5_Premium (31-60$),true,false,509,899912.0,0.77
1_Free,true,false,948,881044.0,0.694
4_Standard (16-30$),true,true,723,705975.0,0.815
6_Luxury (60$+),true,false,33,505606.0,0.695
4_Standard (16-30$),true,false,1793,486015.0,0.773
3_Mid (6-15$),true,true,1864,378927.0,0.799
1_Free,false,true,1465,156993.0,0.732
5_Premium (31-60$),false,true,51,150588.0,0.808


## 10. Profile of top games

Examining the top 20 games by ownership to identify common traits among the most successful titles.

In [0]:
df_top_games = (df
    .select(
        "name", "publisher", "genre", "price_usd",
        "avg_owners", "total_reviews", "rating_ratio",
        "lang_count", "platform_count", "release_year"
    )
    .orderBy(F.desc("avg_owners"))
    .limit(20)
)
display(df_top_games)

name,publisher,genre,price_usd,avg_owners,total_reviews,rating_ratio,lang_count,platform_count,release_year
Dota 2,Valve,"List(Action, Free to Play, Strategy)",0.0,350000000,1852811,0.828414231133127,28,3,2013
Counter-Strike: Global Offensive,Valve,"List(Action, Free to Play)",0.0,75000000,6730438,0.8830547135268165,28,3,2012
Team Fortress 2,Valve,"List(Action, Free to Play)",0.0,75000000,903830,0.9364670347299824,26,3,2007
New World,Amazon Games,"List(Action, Adventure, Massively Multiplayer, RPG)",19.99,75000000,238165,0.6878634560073899,8,1,2021
PUBG: BATTLEGROUNDS,"KRAFTON, Inc.","List(Action, Adventure, Free to Play, Massively Multiplayer)",0.0,75000000,2093876,0.5661084992616564,17,1,2017
War Thunder,Gaijin Distribution KFT,"List(Action, Free to Play, Massively Multiplayer, Simulation)",0.0,35000000,341395,0.7843758695938722,18,3,2013
Path of Exile,Grinding Gear Games,"List(Action, Adventure, Free to Play, Indie, Massively Multiplayer, RPG)",0.0,35000000,192817,0.8713598904660896,7,2,2013
Unturned,Smartly Dressed Games,"List(Action, Adventure, Casual, Free to Play, Indie)",0.0,35000000,493769,0.914741508681185,1,3,2017
Fall Guys: Ultimate Knockout,Ravenscourt,"List(Action, Casual, Indie, Massively Multiplayer, Sports)",0.0,35000000,470671,0.8128310433402526,11,1,2020
Half-Life 2: Lost Coast,Valve,List(Action),0.0,35000000,10842,0.8825862387013467,9,3,2005


## 11. Segmentation: Top 100 / Top 1000 / Rest

Comparing the profile of top games vs the rest reveals what separates blockbusters from average titles.

> Note: Top 1000 is inclusive of Top 100. Rows in the profile table do not sum to the total game count because the segments overlap by design.

In [0]:
df_sorted = df.orderBy(F.desc("avg_owners"), F.asc("appid"))

top_100_ids = [row.appid for row in df_sorted.limit(100).select("appid").collect()]
top_1000_ids = [row.appid for row in df_sorted.limit(1000).select("appid").collect()]

df_top100 = df.filter(F.col("appid").isin(top_100_ids)).select(
    F.lit("Top 100").alias("segment"),
    "price_usd", "lang_count", "rating_ratio", "total_reviews"
)
df_top1000 = df.filter(F.col("appid").isin(top_1000_ids)).select(
    F.lit("Top 1000").alias("segment"),
    "price_usd", "lang_count", "rating_ratio", "total_reviews"
)
df_rest = df.filter(~F.col("appid").isin(top_1000_ids)).select(
    F.lit("Rest").alias("segment"),
    "price_usd", "lang_count", "rating_ratio", "total_reviews"
)

df_segment_profile = (
    df_top100.unionByName(df_top1000).unionByName(df_rest)
    .groupBy("segment")
    .agg(
        F.count("*").alias("games"),
        F.round(F.avg("price_usd"), 2).alias("avg_price"),
        F.round(F.avg("lang_count"), 1).alias("avg_languages"),
        F.round(F.avg("rating_ratio"), 3).alias("avg_rating"),
        F.round(F.avg("total_reviews"), 0).alias("avg_reviews")
    )
)
display(df_segment_profile)

segment,games,avg_price,avg_languages,avg_rating,avg_reviews
Top 100,100,11.08,11.6,0.862,368097.0
Top 1000,1000,16.57,9.2,0.845,72448.0
Rest,54690,7.57,3.5,0.735,419.0


Databricks visualization. Run in Databricks to view.

### 11b. Language profile: Top 20 vs market average

In [0]:
df_top100_languages = (df
    .filter(F.col("appid").isin(top_100_ids))
    .filter(F.col("languages").isNotNull())
    .withColumn("language_single", F.explode(F.col("languages")))
    .groupBy("language_single")
    .agg(
        F.count("*").alias("top100_count"),
        F.round(F.count("*") / 100 * 100, 1).alias("pct_top100")
    )
    .orderBy(F.desc("top100_count"))
    .limit(20)
)
df_all_languages = (df
    .filter(F.col("languages").isNotNull())
    .withColumn("language_single", F.explode(F.col("languages")))
    .groupBy("language_single")
    .agg(F.count("*").alias("all_count"))
)
df_lang_comparison = (df_top100_languages
    .join(df_all_languages, on="language_single", how="left")
    .withColumn("pct_all", F.round(F.col("all_count") / 55680 * 100, 1))
    .withColumn("lift", F.round(F.col("pct_top100") / F.col("pct_all"), 1))
    .orderBy(F.desc("top100_count"))
)
display(df_lang_comparison)

language_single,top20_count,pct_top20,all_count,pct_all,lift
Thai,21,21.0,1699,3.1,6.8
Czech,27,27.0,2352,4.2,6.4
Turkish,40,40.0,3602,6.5,6.2
Polish,54,54.0,5375,9.7,5.6
Korean,63,63.0,6599,11.9,5.3
Swedish,19,19.0,2047,3.7,5.1
Ukrainian,18,18.0,1928,3.5,5.1
Traditional Chinese,56,56.0,6307,11.3,5.0
Portuguese - Brazil,56,56.0,6750,12.1,4.6
Italian,74,74.0,9304,16.7,4.4


Databricks visualization. Run in Databricks to view.

## 12. Category impact (Multiplayer, VR, Achievements...)

Steam categories (Single-player, Multiplayer, Steam Achievements, etc.) may reveal which features correlate with higher ownership.

`categories` has 970 NULLs (1.7% excluded).

In [0]:
df_cat = (df
    .filter(F.col("categories").isNotNull())
    .withColumn("category_single", F.explode(F.col("categories")))
    .groupBy("category_single")
    .agg(
        F.count("*").alias("game_count"),
        F.round(F.avg("avg_owners"), 0).alias("avg_owners"),
        F.round(F.avg("rating_ratio"), 2).alias("avg_rating"),
    )
    .orderBy(F.desc("avg_owners"))
)
display(df_cat)

category_single,game_count,avg_owners,avg_rating
SteamVR Collectibles,40,9131875.0,0.84
Valve Anti-Cheat enabled,111,8638739.0,0.81
Includes Source SDK,44,3221477.0,0.79
Remote Play on Tablet,868,2349510.0,0.87
Remote Play on Phone,714,2049146.0,0.87
In-App Purchases,1349,1324429.0,0.67
Mods,2,1125000.0,0.86
Commentary available,204,1118652.0,0.76
Steam Workshop,1623,979070.0,0.8
MMO,787,917865.0,0.63


Databricks visualization. Run in Databricks to view.

## Conclusion -- Facteurs de popularite sur Steam (Q13)

Analyse sur 55 690 jeux. Metrique principale : `avg_owners` (estimation SteamSpy).

---

### 1. Distribution globale

| Statistique | avg_owners | total_reviews | rating_ratio |
|-------------|-----------|---------------|--------------|
| Moyenne | 133 514 | 1 713 | 0.737 |
| Mediane (P50) | 10 000 | 26 | 0.799 |
| Ecart-type | 1 849 925 | 35 688 | 0.242 |
| P25 | 10 000 | 7 | 0.609 |
| P75 | 35 000 | 145 | 0.925 |
| Max | 350 000 000 | 6 730 438 | 1.0 |

Distribution extremement asymetrique : la mediane (10 000) est 13x inferieure a la moyenne (133 514). Le P25 et la mediane sont tous deux a 10 000, ce qui correspond au plancher du range SteamSpy "0-20 000" (voir Limites). La majorite des jeux restent sous le radar ; le succes est concentre sur une minorite.

### 2. Impact du prix

| Tier | Jeux | Avg owners | Avg reviews | Avg rating |
|------|------|-----------|-------------|------------|
| Free | 7 779 | 314 524 | 3 468 | 0.697 |
| Budget (1-5$) | 23 498 | 44 369 | 285 | 0.724 |
| Mid (6-15$) | 17 750 | 92 806 | 1 120 | 0.760 |
| Standard (16-30$) | 5 497 | 296 577 | 4 814 | 0.765 |
| Premium (31-60$) | 1 044 | 622 495 | 14 422 | 0.769 |
| Luxury (60$+) | 122 | 152 664 | 2 354 | 0.719 |

Le tier Premium (31-60$) domine en owners moyen (622 495), soit 2.1x le Standard (296 577) et 14x le Budget (44 369). Le tier Luxury (60$+) chute a 152 664 owners -- le prix plafond est autour de 60$. Le Free-to-Play se place 2e en volume (314 524) mais avec le rating le plus bas (0.697).

### 3. Impact du genre

Genres avec >= 500 jeux, tries par avg_owners :

| Genre | Jeux | Avg owners | Avg reviews | Avg rating | Avg revenue |
|-------|------|-----------|-------------|------------|-------------|
| Massively Multiplayer | 1 460 | 661 185 | 7 478 | 0.629 | 4 061 752 |
| Free to Play | 3 393 | 580 029 | 6 780 | 0.722 | 29 907 |
| Action | 23 759 | 201 095 | 2 717 | 0.730 | 2 473 019 |
| RPG | 9 534 | 183 200 | 2 381 | 0.731 | 2 850 130 |
| Strategy | 10 895 | 150 173 | 1 450 | 0.719 | 1 849 476 |
| Adventure | 21 431 | 124 114 | 1 649 | 0.739 | 1 737 937 |
| Simulation | 10 836 | 123 071 | 1 659 | 0.694 | 1 732 166 |
| Sports | 2 666 | 120 056 | 1 378 | 0.698 | 1 181 507 |
| Indie | 39 681 | 80 358 | 927 | 0.742 | 815 165 |
| Casual | 22 086 | 58 107 | 524 | 0.743 | 365 886 |

Le MMO domine en owners (661 185) et en revenu moyen (4.1M) mais affiche le rating le plus bas (0.629). Action et RPG offrent le meilleur equilibre volume/rating/revenu. Le genre "Free to Play" a un revenu moyen quasi nul (29 907) car `estimated_revenue = avg_owners * price_usd` et price_usd = 0 pour ces jeux. Note : les revenus moyens par genre incluent les jeux F2P (revenu estime ~0), contrairement au notebook 02_Genre_Analysis qui les exclut, ce qui explique les ecarts observes entre les deux analyses.

### 4. Impact de la plateforme

| Plateformes | Jeux | Avg owners | Avg reviews | Avg rating |
|-------------|------|-----------|-------------|------------|
| 1 (Windows only) | 41 285 | 102 130 | 1 251 | 0.722 |
| 2 (Win + 1) | 7 599 | 137 186 | 1 679 | 0.773 |
| 3 (Win + Mac + Linux) | 6 806 | 319 782 | 4 547 | 0.783 |

Les jeux 3 plateformes ont 3.1x plus d'owners que les jeux Windows-only (319 782 vs 102 130) et un meilleur rating (0.783 vs 0.722). Attention : ceci est probablement un effet de selection -- les studios capables de porter sur 3 OS sont aussi ceux qui investissent plus globalement dans la qualite.

### 5. Impact des langues

Tiers utilises : 1 / 2-5 / 6-10 / 11+ (section 5).

| Tier langues | Jeux | Avg owners | Avg reviews | Avg rating |
|-------------|------|-----------|-------------|------------|
| Single (1) | 29 655 | 48 332 | 319 | 0.724 |
| Few (2-5) | 15 062 | 79 516 | 646 | 0.740 |
| Good (6-10) | 6 384 | 277 201 | 3 260 | 0.761 |
| Extensive (11+) | 4 579 | 662 729 | 12 097 | 0.771 |

Le facteur le plus discriminant : les jeux 11+ langues ont 13.7x plus d'owners que les jeux monolingues (662 729 vs 48 332). Le saut principal se situe au passage de 5 a 10 langues (79 516 -> 277 201, soit x3.5).

Note : la section 9 (croisement multi-facteurs) utilise `has_multi_lang = lang_count >= 5` comme seuil, differemment des tiers ci-dessus. Ce seuil a 5 inclut la borne superieure du tier Few (5 langues) plus les tiers Good (6-10) et Extensive (11+).

### 6. Impact temporel (release_year >= 2015)

| Annee | Jeux | Avg owners | Avg reviews | Avg rating |
|-------|------|-----------|-------------|------------|
| 2015 | 2 575 | 248 384 | 3 654 | 0.690 |
| 2016 | 4 185 | 162 806 | 2 017 | 0.706 |
| 2017 | 6 017 | 123 793 | 1 698 | 0.700 |
| 2018 | 7 678 | 84 989 | 1 259 | 0.695 |
| 2019 | 6 968 | 73 813 | 997 | 0.728 |
| 2020 | 8 305 | 75 845 | 1 163 | 0.756 |
| 2021 | 8 823 | 62 021 | 689 | 0.766 |
| 2022 | 7 455 | 46 911 | 529 | 0.794 |

Baisse continue des avg_owners : de 248 384 (2015) a 46 911 (2022), soit une division par 5.3. Deux facteurs conjugues : le biais temporel (les jeux anciens accumulent plus d'owners mecaniquement) et la saturation du marche (le nombre de sorties a triple entre 2015 et 2021). A noter : le rating moyen augmente de 0.690 a 0.794, suggerant une meilleure qualite percue des sorties recentes malgre moins d'owners.

### 7. Top publishers (>= 20 jeux, top 15 par avg_owners)

| Publisher | Jeux | Avg owners | Avg reviews | Rating | Revenue total |
|-----------|------|-----------|-------------|--------|---------------|
| Valve | 35 | 22 939 857 | 357 381 | 0.891 | 2 400 480 000 |
| CAPCOM Co., Ltd. | 24 | 2 360 208 | 39 790 | 0.833 | 1 674 802 050 |
| Bethesda Softworks | 55 | 2 042 273 | 41 789 | 0.793 | 2 210 216 850 |
| Xbox Game Studios | 46 | 1 945 978 | 37 146 | 0.860 | 2 259 672 350 |
| Electronic Arts | 88 | 1 739 602 | 27 701 | 0.797 | 3 188 435 000 |
| Warner Bros. Interactive Entertainment | 29 | 1 431 552 | 16 404 | 0.824 | 622 441 850 |
| **Ubisoft** | **127** | **1 358 780** | **23 613** | **0.776** | **3 480 300 400** |
| 2K | 67 | 1 175 896 | 12 758 | 0.713 | 1 301 797 150 |
| Bohemia Interactive | 28 | 1 113 214 | 22 137 | 0.681 | 736 388 100 |
| Activision | 51 | 985 980 | 9 433 | 0.782 | 1 678 360 750 |
| Paradox Interactive | 69 | 978 406 | 16 691 | 0.745 | 1 868 884 600 |
| Deep Silver | 38 | 920 132 | 11 927 | 0.679 | 603 975 800 |
| Konami Digital Entertainment | 25 | 773 200 | 11 528 | 0.727 | 266 348 700 |
| Capcom | 35 | 759 571 | 9 269 | 0.755 | 604 792 050 |
| Team17 | 27 | 699 815 | 12 371 | 0.821 | 432 559 050 |

**Ubisoft** : 7e par avg_owners (1 358 780) mais **1er en revenu total** (3.48 Md) grace au plus grand catalogue (127 jeux). Le rating (0.776) est inferieur a Valve (0.891), CAPCOM (0.833) et Xbox Game Studios (0.860). Levier d'amelioration : la qualite percue, pas le volume.

### 8. Free-to-Play vs Paid

| Modele | Jeux | Avg owners | Avg reviews | Avg rating | Owners P90 |
|--------|------|-----------|-------------|------------|------------|
| Paid | 47 911 | 104 124 | 1 428 | 0.743 | 150 000 |
| Free to Play | 7 779 | 314 524 | 3 468 | 0.697 | 350 000 |

Le F2P attire 3x plus d'owners en moyenne (314 524 vs 104 124) mais avec un rating inferieur (0.697 vs 0.743). Au P90, l'ecart se reduit : 350 000 F2P vs 150 000 Paid. Le F2P concentre la masse dans les titres etablis ; la longue traine F2P reste modeste.

### 9. Croisement multi-facteurs (top 5 segments)

Seuils utilises : `has_multi_lang = lang_count >= 5`, `has_multi_platform = platform_count >= 2`.

| Tier prix | Multi-lang | Multi-platform | Jeux | Avg owners | Avg rating |
|-----------|-----------|----------------|------|-----------|------------|
| Premium (31-60$) | Oui | Oui | 85 | 1 904 176 | 0.790 |
| Free | Oui | Oui | 538 | 1 696 171 | 0.685 |
| Premium (31-60$) | Oui | Non | 509 | 899 912 | 0.770 |
| Free | Oui | Non | 948 | 881 044 | 0.694 |
| Standard (16-30$) | Oui | Oui | 723 | 705 975 | 0.815 |

Le segment le plus performant : **Premium + multi-langue + multi-plateforme** (1 904 176 owners, rating 0.790) -- mais seulement 85 jeux, donc forte variance. Le segment **Standard + multi-langue + multi-plateforme** (723 jeux, 705 975 owners, rating 0.815) est plus robuste statistiquement et affiche le meilleur rating du top 5.

### 10. Profil des top 20 jeux

Traits communs des 20 jeux les plus possedes :
- 13/20 sont Free-to-Play (Dota 2, CS:GO, TF2, PUBG, War Thunder, Path of Exile, etc.)
- Le n.1 est Dota 2 (350M owners, Valve)
- 7 jeux payes : New World (19.99$), Terraria (9.99$), Garry's Mod (9.99$), ELDEN RING (59.99$), Rust (39.99$), GTA V (29.98$), PAYDAY 2 (9.99$)
- Valve place 5 jeux dans le top 20
- Nombre moyen de langues dans le top 20 : 14.7, nombre moyen de plateformes : 1.95

### 11. Segmentation Top 100 / Top 1000 / Reste

| Segment|Jeux|Prix moyen|Langues moy.|Rating moy.|Reviews moy.|
|---|---|---|---|---|---|
|Top 100|100|11.08$|11.6|0.862|368097|
|Top 1000|1000|16.57$|9.2|0.845|72448|
|Rest|54690|7.57$|3.5|0.735|419|

Ce qui separe les blockbusters du reste : le support linguistique (11.6 langues dans le Top 100 vs 3.5 pour le reste, soit x3.3) et la qualite percue (rating 0.862 vs 0.735). Le prix moyen du Top 100 (11.08$) est inferieur au Top 1000 (16.57$) car beaucoup de F2P figurent dans le Top 100. L'ecart en reviews est massif : x879 entre Top 100 et le reste.

### 12. Categories Steam

Top 10 des categories par avg_owners :

|category_single|game_count|avg_owners|avg_rating|
|---|---|---|---|
|SteamVR Collectibles|40|9131875|0.84|
|Valve Anti-Cheat enabled|111|8638739|0.81|
|Includes Source SDK|44|3221477|0.79|
|Remote Play on Tablet|868|2349510|0.87|
|Remote Play on Phone|714|2049146|0.87|
|In-App Purchases|1349|1324429|0.67|
|Mods|2|1125000|0.86|
|Commentary available|204|1118652|0.76|
|Steam Workshop|1623|979070|0.8|
|MMO|787|917865|0.63|

Les 5 premieres categories sont portees par un faible nombre de jeux (40 a 868), souvent des titres Valve (SteamVR Collectibles, Valve Anti-Cheat, Source SDK). Remote Play on Tablet et Remote Play on Phone affichent les meilleurs ratings (0.87) et des avg_owners superieurs a 2M, refletant la presence de blockbusters dans ces petites populations. In-App Purchases (1 324 429 owners) et MMO (917 865 owners) confirment le paradoxe popularite/satisfaction avec les pires ratings (0.67 et 0.63). Steam Workshop (979 070, rating 0.80) reste un marqueur de retention et de communaute.

---

### Synthese pour Ubisoft

Ubisoft se positionne 7e publisher par avg_owners (1.36M) et 1er par revenu total estime (3.48 Md) sur 127 jeux. Son rating moyen (0.776) est en retrait par rapport aux references (Valve 0.891, Xbox 0.860, CAPCOM 0.833).

**Recommandations factuelles basees sur les donnees :**

| Facteur | Constat | Recommandation |
|---------|---------|----------------|
| Genre | Action (201k owners, 2.5M rev) et RPG (183k owners, 2.9M rev) dominent en volume et revenu | Maintenir le focus Action-RPG/Action-Adventure |
| Prix | Le tier 31-60$ maximise les owners (622k) ; au-dela de 60$ le volume chute a 153k | Positionner entre 30$ et 60$ |
| Langues | 11+ langues = 662 729 owners (x13.7 vs monolingue) ; Top 100 = 11.6 langues en moyenne | Viser 11+ langues |
| Plateformes | 3 plateformes = 319 782 owners (x3.1 vs Windows-only) | Windows obligatoire, Mac/Linux si le budget le permet |
| Categories | Co-op (511k), Steam Workshop (979k), Online Co-op (645k) correlent avec plus d'owners | Integrer un mode cooperatif et le support Workshop |
| Rating | L'ecart Ubisoft (0.776) vs Top 100 (0.862) represente un levier sous-exploite | Prioriser la qualite percue (rating) plutot que le volume de sorties |

### Limites de l'analyse

1. **Plancher SteamSpy** : la mediane de `avg_owners` est exactement 10 000, qui est le minimum du range SteamSpy "0-20 000". Le P25 est aussi a 10 000. La vraie mediane est probablement inferieure a 10 000 -- on ne peut pas distinguer un jeu a 500 owners d'un jeu a 9 999 owners dans ces donnees.

2. **Biais temporel cumulatif** : `avg_owners` est un snapshot cumulatif, pas un flux annuel. Un jeu de 2015 a eu 7 ans de plus qu'un jeu de 2022 pour accumuler des owners. La baisse apparente de 248 384 (2015) a 46 911 (2022) est en partie mecanique, pas uniquement due a la saturation du marche.

3. **Proxy revenue** : `estimated_revenue = avg_owners * price_usd`. Ce n'est pas du chiffre d'affaires reel : il ignore les soldes (le prix paye est souvent inferieur au prix affiche), les remboursements, les bundles, et les revenus in-app. Pour les jeux F2P, ce proxy donne ~0 alors que leur monetisation reelle (microtransactions) peut etre massive.

4. **Correlation, pas causalite** : le multi-plateforme et le multi-langue correlent avec plus d'owners, mais les studios qui investissent dans ces dimensions sont aussi ceux avec plus de budget et de marketing. L'effet propre de chaque facteur n'est pas isolable sans modele causal.

5. **Seuils variables** : la section 5 utilise des tiers (1, 2-5, 6-10, 11+) pour le nombre de langues, tandis que la section 9 utilise un seuil binaire `lang_count >= 5` pour `has_multi_lang`. Les chiffres ne sont pas directement comparables entre ces deux sections.

6. **Couverture des metriques** : `rating_ratio` couvre 99.7% du dataset (163 NULLs), `genre` 99.7% (160 NULLs), `categories` 98.3% (970 NULLs). Ces taux sont suffisants mais pas parfaits.